In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import joblib

# Đọc bộ dữ liệu 253.000 dòng của CDC
df = pd.read_csv('../data/raw/heart_disease_health_indicators_BRFSS2015.csv')

print("Kích thước dữ liệu:", df.shape)
# Kiểm tra xem có cột nào bị thiếu dữ liệu không (sẽ in ra 0 hết)
print("Số lượng dữ liệu thiếu:\n", df.isnull().sum().sum())
df.head()

Kích thước dữ liệu: (253680, 22)
Số lượng dữ liệu thiếu:
 0


,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [2]:
# Tách dữ liệu: y là đáp án (Cột Đột quỵ), X là các manh mối còn lại
X = df.drop('Stroke', axis=1)
y = df['Stroke']

# Vì 253.000 dòng là quá lớn, để AI học nhanh trên máy cá nhân, 
# ta sẽ cắt bớt lấy 100.000 dòng ngẫu nhiên. (Nếu máy bạn khỏe, có thể bỏ 3 dòng này)
df_sample = df.sample(n=100000, random_state=42)
X = df_sample.drop('Stroke', axis=1)
y = df_sample['Stroke']

print("Tỉ lệ Bệnh/Khỏe ban đầu:\n", y.value_counts())

Tỉ lệ Bệnh/Khỏe ban đầu:
 Stroke
0.0    95921
1.0     4079
Name: count, dtype: int64


In [3]:
print("Đang nhân bản dữ liệu bệnh nhân bằng SMOTE. Vui lòng đợi khoảng 10-30 giây...")

smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X, y)

print("✅ Đã cân bằng xong! Tỉ lệ mới:\n", y_balanced.value_counts())

Đang nhân bản dữ liệu bệnh nhân bằng SMOTE. Vui lòng đợi khoảng 10-30 giây...
✅ Đã cân bằng xong! Tỉ lệ mới:
 Stroke
0.0    95921
1.0    95921
Name: count, dtype: int64


In [4]:
# Chia 80% để học, 20% để thi
X_train, X_test, y_train, y_test = train_test_split(X_balanced, y_balanced, test_size=0.2, random_state=42)

print(f"Đang dạy AI trên {X_train.shape[0]} mẫu bệnh án...")

# Giả sử cột Rượu Bia nằm ở vị trí thứ 11 trong tập dữ liệu của bạn
rang_buoc = (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)

# Khởi tạo não bộ XGBoost (Cấu hình mạnh hơn một chút để xử lý dữ liệu lớn)
model = xgb.XGBClassifier(
    n_estimators=200,      # Đọc 200 cuốn sách quy luật
    max_depth=5,           # Mỗi cuốn sách phân tích sâu 5 tầng logic
    learning_rate=0.1,     # Tốc độ học chậm mà chắc
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    monotone_constraints=rang_buoc # Tiêm kiến thức y khoa vào đây!
)

# Bắt đầu học!
model.fit(X_train, y_train)

print("✅ Quá trình huấn luyện hoàn tất!")

Đang dạy AI trên 153473 mẫu bệnh án...


d:\LuanVanTotNghiep\Test\AI-Stroke-Prediction\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [08:22:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Quá trình huấn luyện hoàn tất!


In [5]:
import lightgbm as lgb
import time

print("🚀 Bắt đầu huấn luyện LightGBM...")
thoi_gian_bat_dau = time.time()

# 1. Khởi tạo "Bộ não" LightGBM
# Các tham số này đã được tinh chỉnh cơ bản cho bài toán phân loại nhị phân
lgbm_model = lgb.LGBMClassifier(
    n_estimators=200,        # Đọc 200 cuốn sách quy luật (Giống XGBoost)
    learning_rate=0.1,       # Tốc độ học
    num_leaves=31,           # ĐẶC SẢN CỦA LIGHTGBM: Số lượng lá tối đa trên mỗi cây (Mặc định 31 là rất tốt)
    max_depth=-1,            # Không giới hạn độ sâu (Để num_leaves tự kiểm soát)
    class_weight='balanced', # Tự động phạt nặng nếu AI đoán sai người bệnh (Hỗ trợ thêm cho SMOTE)
    random_state=42,
    n_jobs=-1                # Huy động 100% sức mạnh CPU của máy tính để chạy nhanh nhất
)

# 2. Cho AI đi học
lgbm_model.fit(X_train, y_train)

thoi_gian_ket_thuc = time.time()
print(f"✅ Huấn luyện hoàn tất! Thời gian chạy: {thoi_gian_ket_thuc - thoi_gian_bat_dau:.2f} giây")

🚀 Bắt đầu huấn luyện LightGBM...
[LightGBM] [Info] Number of positive: 76703, number of negative: 76770
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007168 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5355
[LightGBM] [Info] Number of data points in the train set: 153473, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
✅ Huấn luyện hoàn tất! Thời gian chạy: 1.29 giây


In [ ]:
# Cho AI làm bài thi cuối kỳ
y_pred = model.predict(X_test)

print("=== KẾT QUẢ ĐÁNH GIÁ AI TRÊN BỘ DỮ LIỆU CDC ===\n")
print(f"Độ chính xác tổng thể (Accuracy): {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Báo cáo chi tiết (Classification Report):")
print(classification_report(y_test, y_pred))

# Lưu não bộ thành file mới
joblib.dump(model, '../xgboost_cdc_stroke_model.pkl')
print("\nĐã lưu mô hình AI mới thành file 'xgboost_cdc_stroke_model.pkl'!")

=== KẾT QUẢ ĐÁNH GIÁ AI TRÊN BỘ DỮ LIỆU CDC ===

Độ chính xác tổng thể (Accuracy): 97.56%

Báo cáo chi tiết (Classification Report):
              precision    recall  f1-score   support

         0.0       0.95      1.00      0.98     19151
         1.0       1.00      0.95      0.98     19218

    accuracy                           0.98     38369
   macro avg       0.98      0.98      0.98     38369
weighted avg       0.98      0.98      0.98     38369


Đã lưu mô hình AI mới thành file 'xgboost_cdc_stroke_model.pkl'!


In [7]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# 1. Yêu cầu AI dự đoán trên tập dữ liệu kiểm tra (X_test)
y_pred = model.predict(X_test)               # AI chốt hạ: 0 (Khỏe) hoặc 1 (Bệnh)
y_pred_proba = model.predict_proba(X_test)[:, 1] # Xác suất phần trăm rủi ro

# 2. Bóc tách Ma trận nhầm lẫn để tính Specificity
# tn: True Negative (Đoán khỏe đúng), fp: False Positive (Báo động giả)
# fn: False Negative (Bỏ lọt bệnh), tp: True Positive (Đoán bệnh đúng)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# 3. Tính toán 6 chỉ số y hệt như trong ảnh
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred) # Recall chính là Sensitivity
specificity = tn / (tn + fp)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

# 4. In ra bảng kết quả cực đẹp để copy vào Word/Excel
ket_qua = pd.DataFrame({
    'Metric': ['Accuracy', 'AUC', 'F1-Score', 'Sensitivity', 'Specificity', 'Precision'],
    'Điểm số': [f"{accuracy*100:.2f}%", f"{auc*100:.2f}%", f"{f1*100:.2f}%", 
                f"{sensitivity*100:.2f}%", f"{specificity*100:.2f}%", f"{precision*100:.2f}%"]
})

print("📊 BẢNG ĐÁNH GIÁ NĂNG LỰC AI DỰ ĐOÁN ĐỘT QUỴ")
print("-" * 45)
print(ket_qua.to_string(index=False))

📊 BẢNG ĐÁNH GIÁ NĂNG LỰC AI DỰ ĐOÁN ĐỘT QUỴ
---------------------------------------------
     Metric Điểm số
   Accuracy  97.56%
        AUC  99.18%
   F1-Score  97.50%
Sensitivity  95.17%
Specificity  99.96%
  Precision  99.96%


In [8]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# 1. Yêu cầu AI dự đoán trên tập dữ liệu kiểm tra (X_test) lightgbm
y_pred = lgbm_model.predict(X_test)               # AI chốt hạ: 0 (Khỏe) hoặc 1 (Bệnh)
y_pred_proba = lgbm_model.predict_proba(X_test)[:, 1] # Xác suất phần trăm rủi ro

# 2. Bóc tách Ma trận nhầm lẫn để tính Specificity
# tn: True Negative (Đoán khỏe đúng), fp: False Positive (Báo động giả)
# fn: False Negative (Bỏ lọt bệnh), tp: True Positive (Đoán bệnh đúng)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# 3. Tính toán 6 chỉ số y hệt như trong ảnh
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred) # Recall chính là Sensitivity
specificity = tn / (tn + fp)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

# 4. In ra bảng kết quả cực đẹp để copy vào Word/Excel
ket_qua = pd.DataFrame({
    'Metric': ['Accuracy', 'AUC', 'F1-Score', 'Sensitivity', 'Specificity', 'Precision'],
    'Điểm số': [f"{accuracy*100:.2f}%", f"{auc*100:.2f}%", f"{f1*100:.2f}%", 
                f"{sensitivity*100:.2f}%", f"{specificity*100:.2f}%", f"{precision*100:.2f}%"]
})

print("📊 BẢNG ĐÁNH GIÁ NĂNG LỰC AI DỰ ĐOÁN ĐỘT QUỴ")
print("-" * 45)
print(ket_qua.to_string(index=False))

📊 BẢNG ĐÁNH GIÁ NĂNG LỰC AI DỰ ĐOÁN ĐỘT QUỴ
---------------------------------------------
     Metric Điểm số
   Accuracy  97.74%
        AUC  99.19%
   F1-Score  97.69%
Sensitivity  95.55%
Specificity  99.94%
  Precision  99.93%


In [9]:
!pip install catboost


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import time
from catboost import CatBoostClassifier

print("🐅 Bắt đầu huấn luyện CatBoost...")
thoi_gian_bat_dau = time.time()

# 1. Khởi tạo não bộ CatBoost
cat_model = CatBoostClassifier(
    iterations=200,             # Đọc 200 cuốn sách quy luật
    learning_rate=0.1,          # Tốc độ học
    depth=6,                    # Độ sâu của cây quyết định
    auto_class_weights='Balanced', # TÍNH NĂNG XỊN: Tự động phạt lỗi nặng nếu đoán sai người bệnh (Cân bằng y tế)
    random_seed=42,
    verbose=0                   # Tắt các dòng log chạy rào rào cho màn hình đỡ rối
)

# 2. Bắt đầu học (Dùng luôn tập X_train, y_train từ đầu)
cat_model.fit(X_train, y_train)

thoi_gian_ket_thuc = time.time()
print(f"✅ Huấn luyện CatBoost hoàn tất! Thời gian: {thoi_gian_ket_thuc - thoi_gian_bat_dau:.2f} giây")

🐅 Bắt đầu huấn luyện CatBoost...
✅ Huấn luyện CatBoost hoàn tất! Thời gian: 4.59 giây


In [11]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# 1. Bắt CatBoost làm bài thi trên đề thi X_test
y_pred_cat = cat_model.predict(X_test)
y_pred_proba_cat = cat_model.predict_proba(X_test)[:, 1]

# 2. Tính toán ma trận nhầm lẫn
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_cat).ravel()

# 3. Tính 6 chỉ số y khoa
accuracy = accuracy_score(y_test, y_pred_cat)
precision = precision_score(y_test, y_pred_cat)
sensitivity = recall_score(y_test, y_pred_cat)
specificity = tn / (tn + fp)
f1 = f1_score(y_test, y_pred_cat)
auc = roc_auc_score(y_test, y_pred_proba_cat)

# 4. In bảng điểm
ket_qua_cat = pd.DataFrame({
    'Metric': ['Accuracy', 'AUC', 'F1-Score', 'Sensitivity', 'Specificity', 'Precision'],
    'Điểm CatBoost': [f"{accuracy*100:.2f}%", f"{auc*100:.2f}%", f"{f1*100:.2f}%", 
                      f"{sensitivity*100:.2f}%", f"{specificity*100:.2f}%", f"{precision*100:.2f}%"]
})

print("📊 BẢNG ĐÁNH GIÁ NĂNG LỰC CATBOOST")
print("-" * 40)
print(ket_qua_cat.to_string(index=False))

📊 BẢNG ĐÁNH GIÁ NĂNG LỰC CATBOOST
----------------------------------------
     Metric Điểm CatBoost
   Accuracy        97.70%
        AUC        99.20%
   F1-Score        97.65%
Sensitivity        95.42%
Specificity        99.98%
  Precision        99.98%


In [13]:
import joblib
import os

print("📦 Đang tiến hành đóng gói các mô hình AI...")



# 2. Lưu mô hình XGBoost (Biến của bạn tên là 'model')
joblib.dump(model, '../trained_models/xgboost_cdc_stroke_model.pkl')
print("✅ Đã lưu thành công: XGBoost")

# 3. Lưu mô hình LightGBM (Biến của bạn tên là 'lgbm_model')
# (Nếu bạn chưa chạy thành công LightGBM thì có thể thêm dấu # ở trước dòng này để ẩn nó đi)
joblib.dump(lgbm_model, '../trained_models/lightgbm_cdc_stroke_model.pkl')
print("✅ Đã lưu thành công: LightGBM")

# 4. Lưu mô hình CatBoost (Biến của bạn tên là 'cat_model')
# (Nếu bạn chưa chạy thành công CatBoost thì có thể thêm dấu # ở trước dòng này để ẩn nó đi)
joblib.dump(cat_model, '../trained_models/catboost_cdc_stroke_model.pkl')
print("✅ Đã lưu thành công: CatBoost")

print("🎉 Hoàn tất! Hãy kiểm tra thư mục 'trained_models' bên trái màn hình nhé.")

📦 Đang tiến hành đóng gói các mô hình AI...
✅ Đã lưu thành công: XGBoost
✅ Đã lưu thành công: LightGBM
✅ Đã lưu thành công: CatBoost
🎉 Hoàn tất! Hãy kiểm tra thư mục 'trained_models' bên trái màn hình nhé.
